# Figure S2 SIMBA Metric Distances

This notebook uses a scalar SIMBA metric throughout, rather than trial-window means from the snips.

Supported metric modes:
- `median_balance`: uses `simba_median_balance_score`
- `mean_zscore_auc`: uses `simba_mean_zscore_auc`

The main output is the animal-by-animal distance matrix and downstream visualizations.

In [ ]:
import sys

import warnings

from pathlib import Path



import dill

import numpy as np

import pandas as pd

import matplotlib.pyplot as plt

import seaborn as sns

from scipy.ndimage import gaussian_filter1d

from scipy.spatial.distance import cdist

from sklearn.manifold import MDS

from trompy import save_figure_atomic



sys.path.insert(0, str(Path("../src").resolve()))



from pickle_compat import enable_dill_pathlib_compat

from figure_config import configure_matplotlib, COLORS, DATAFOLDER, FIGSFOLDER



enable_dill_pathlib_compat()

configure_matplotlib()



SIMBA_SCALAR_METRIC = "median_balance"

DISTANCE_METRIC = "euclidean"

APPLY_TRIAL_SMOOTHING = True

SMOOTH_SIGMA = 1.2

SAVE_FIGS = False



GROUP_ORDER = [

    ("replete", "10NaCl"),

    ("replete", "45NaCl"),

    ("deplete", "10NaCl"),

    ("deplete", "45NaCl"),

]

GROUP_LABELS = {

    ("replete", "10NaCl"): "Replete 10NaCl",

    ("replete", "45NaCl"): "Replete 45NaCl",

    ("deplete", "10NaCl"): "Deplete 10NaCl",

    ("deplete", "45NaCl"): "Deplete 45NaCl",

}

GROUP_COLORS = {group: COLORS[idx] for idx, group in enumerate(GROUP_ORDER)}

In [ ]:
assembled_data_path = DATAFOLDER / "assembled_data.pickle"

with open(assembled_data_path, "rb") as f:
    data = dill.load(f)

x_array = data["x_array"].copy()
params = data.get("params", {})
metadata = data.get("metadata", {})

metric_options = {
    "median_balance": {
        "column": "simba_median_balance",
        "label": "SIMBA median-balance score",
        "file_stub": "median_balance",
    },
    "mean_zscore_auc": {
        "column": "simba_zscore_mean",
        "label": "SIMBA mean z-score AUC",
        "file_stub": "mean_zscore_auc",
    },
}

if SIMBA_SCALAR_METRIC not in metric_options:
    raise ValueError(f"Unknown SIMBA_SCALAR_METRIC: {SIMBA_SCALAR_METRIC}")

metric_config = metric_options[SIMBA_SCALAR_METRIC]
metric_col = metric_config["column"]
metric_label = metric_config["label"]
metric_file_stub = metric_config["file_stub"]

required_cols = ["id", "condition", "infusiontype", "trial", metric_col]
missing_cols = [col for col in required_cols if col not in x_array.columns]
if missing_cols:
    raise KeyError(f"x_array is missing required columns: {missing_cols}")

print(f"Loaded {assembled_data_path}")
print(f"Using metric column: {metric_col}")
print(f"Metric label: {metric_label}")
print(f"Rows in x_array: {len(x_array)}")
print(f"Metric range: {x_array[metric_col].min():.4f} to {x_array[metric_col].max():.4f}")

In [ ]:
scalar_df = x_array.loc[:, ["id", "condition", "infusiontype", "trial", metric_col]].copy()
scalar_df = scalar_df.dropna(subset=[metric_col])
scalar_df["group_key"] = list(zip(scalar_df["condition"], scalar_df["infusiontype"]))
scalar_df = scalar_df[scalar_df["group_key"].isin(GROUP_ORDER)].copy()
scalar_df["group_key"] = pd.Categorical(scalar_df["group_key"], categories=GROUP_ORDER, ordered=True)
scalar_df = scalar_df.sort_values(["group_key", "id", "trial"])

trial_counts = scalar_df.groupby(["group_key", "id"]).trial.nunique().sort_index()
print("Trials per animal/group series:")
print(trial_counts.groupby(level=0).agg(["count", "min", "max"]))

metric_matrix_df = scalar_df.pivot_table(
    index=["group_key", "id"],
    columns="trial",
    values=metric_col,
    aggfunc="mean",
).sort_index()

if metric_matrix_df.isna().any().any():
    missing_by_series = metric_matrix_df.isna().sum(axis=1)
    missing_by_series = missing_by_series[missing_by_series > 0]
    raise ValueError(
        "Some animal/group series are missing trial values. Missing counts:\n"
        f"{missing_by_series.to_string()}"
    )

metric_matrix = metric_matrix_df.to_numpy(dtype=float)
if APPLY_TRIAL_SMOOTHING:
    metric_matrix_for_distance = gaussian_filter1d(metric_matrix, sigma=SMOOTH_SIGMA, axis=1)
else:
    metric_matrix_for_distance = metric_matrix.copy()

series_index = list(metric_matrix_df.index)
series_labels = [f"{GROUP_LABELS[group_key]} | {animal_id}" for group_key, animal_id in series_index]
group_names = [GROUP_LABELS[group_key] for group_key, _ in series_index]
group_to_row_indices = {
    group_key: [idx for idx, (series_group, _) in enumerate(series_index) if series_group == group_key]
    for group_key in GROUP_ORDER
}
group_sizes = {GROUP_LABELS[group_key]: len(indices) for group_key, indices in group_to_row_indices.items()}
group_boundaries = np.cumsum([len(group_to_row_indices[group_key]) for group_key in GROUP_ORDER])

print(f"Metric matrix shape: {metric_matrix.shape}")
print(f"Distance input shape: {metric_matrix_for_distance.shape}")
print("Series per group:")
print(group_sizes)

metric_matrix_df.head()

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(7, 1.8), sharey=True,
                         gridspec_kw={"width_ratios": [1, 1, 1, 1, 0.1],
                                      "left": 0.1, "right": 0.95, "bottom": 0.3, "wspace": 0.1})
trial_numbers = metric_matrix_df.columns.to_numpy()

for ax, group_key in zip(axes[:4], GROUP_ORDER):
    row_indices = group_to_row_indices[group_key]
    group_series = metric_matrix[row_indices]
    color = GROUP_COLORS[group_key]

    for series in group_series:
        ax.plot(trial_numbers, series, color=color, alpha=0.2, linewidth=1)

    ax.plot(trial_numbers, np.nanmean(group_series, axis=0), color=color, linewidth=2)
    ax.axhline(0, color="gray", linestyle="--", linewidth=1)
    sns.despine(ax=ax, offset=5)
    ax.set_xlabel("Trial")

axes[0].set_ylabel(metric_label)
axes[4].axis("off")

if SAVE_FIGS:
    save_figure_atomic(fig, f"figure_S2_simba_{metric_file_stub}_trajectories", folder=FIGSFOLDER)


In [ ]:
distances = cdist(metric_matrix_for_distance, metric_matrix_for_distance, metric=DISTANCE_METRIC)
distances_df = pd.DataFrame(distances, index=series_labels, columns=series_labels)

group_distance_matrix = pd.DataFrame(
    index=[GROUP_LABELS[group_key] for group_key in GROUP_ORDER],
    columns=[GROUP_LABELS[group_key] for group_key in GROUP_ORDER],
    dtype=float,
)

for row_group in GROUP_ORDER:
    row_idx = group_to_row_indices[row_group]
    for col_group in GROUP_ORDER:
        col_idx = group_to_row_indices[col_group]
        group_distance_matrix.loc[GROUP_LABELS[row_group], GROUP_LABELS[col_group]] = distances[np.ix_(row_idx, col_idx)].mean()

print(f"Distance metric: {DISTANCE_METRIC}")
print(f"Mean off-diagonal distance: {distances[~np.eye(distances.shape[0], dtype=bool)].mean():.4f}")
group_distance_matrix

In [ ]:
heatmap_vmin = np.nanpercentile(distances, 5)
heatmap_vmax = np.nanpercentile(distances, 95)
cmap = plt.get_cmap("Oranges").reversed()

# All-animal heatmap
fig, ax = plt.subplots(figsize=(1.8, 1.8))

sns.heatmap(
    distances,
    ax=ax,
    cmap=cmap,
    vmin=heatmap_vmin,
    vmax=heatmap_vmax,
    cbar=False,
    xticklabels=False,
    yticklabels=False,
)

for boundary in group_boundaries[:-1]:
    ax.axhline(boundary, color="white", linewidth=1)
    ax.axvline(boundary, color="white", linewidth=1)

if SAVE_FIGS:
    save_figure_atomic(fig, f"figure_S2_simba_{metric_file_stub}_distance_heatmap_all", folder=FIGSFOLDER)

# Group-averaged heatmap with separate colorbar
fig2, ax2 = plt.subplots(figsize=(1.8, 1.8))
fig2_cbar, ax2_cbar = plt.subplots(figsize=(0.18, 1.8))

sns.heatmap(
    group_distance_matrix,
    ax=ax2,
    cmap=cmap,
    annot=True,
    fmt=".2f",
    cbar_ax=ax2_cbar,
    linewidths=1,
    linecolor="white",
)
ax2.set_yticks([])
ax2.set_xticks([])

if SAVE_FIGS:
    save_figure_atomic(fig2, f"figure_S2_simba_{metric_file_stub}_distance_heatmap", folder=FIGSFOLDER)
    save_figure_atomic(fig2_cbar, f"figure_S2_simba_{metric_file_stub}_distance_heatmap_colorbar", folder=FIGSFOLDER)

plt.show()


In [ ]:
distance_scale = np.nanmax(distances)

if distance_scale > 0:
    distances_for_mds = distances / distance_scale
else:
    distances_for_mds = distances.copy()

with warnings.catch_warnings():
    warnings.filterwarnings("ignore", category=FutureWarning, module="sklearn.manifold._mds")
    mds = MDS(
        n_components=2,
        metric=True,
        dissimilarity="precomputed",
        random_state=42,
        n_init=8,
        max_iter=500,
        init="random",
    )
    coords = mds.fit_transform(distances_for_mds)

fig, ax = plt.subplots(figsize=(1.8, 1.8),
                       gridspec_kw={"left": 0.15, "bottom": 0.15})

group_centroids = []
for group_key in GROUP_ORDER:
    row_idx = group_to_row_indices[group_key]
    group_coords = coords[row_idx]
    color = GROUP_COLORS[group_key]
    centroid = group_coords.mean(axis=0)
    group_centroids.append(centroid)

    for coord in group_coords:
        ax.plot([coord[0], centroid[0]], [coord[1], centroid[1]], color=color, alpha=0.3, zorder=0)

    ax.scatter(group_coords[:, 0], group_coords[:, 1], s=40, edgecolor=color, facecolor="none")
    ax.scatter(centroid[0], centroid[1], s=100, color=color)

ax.set_xlabel("MDS1")
ax.set_ylabel("MDS2")
ax.set_xticks([])
ax.set_yticks([])
sns.despine(ax=ax, offset=5)

if SAVE_FIGS:
    save_figure_atomic(fig, f"figure_S2_simba_{metric_file_stub}_distance_mds", folder=FIGSFOLDER)

plt.show()


In [ ]:
nearest_neighbors = []
for idx, label in enumerate(series_labels):
    row = distances[idx].copy()
    row[idx] = np.inf
    neighbor_idx = int(np.argmin(row))
    nearest_neighbors.append({
        "series": label,
        "group": group_names[idx],
        "nearest_neighbor": series_labels[neighbor_idx],
        "nearest_neighbor_group": group_names[neighbor_idx],
        "distance": row[neighbor_idx],
    })

nearest_neighbors_df = pd.DataFrame(nearest_neighbors).sort_values(["group", "distance"])
nearest_neighbors_df.head(20)